# preRA cohort scRNA analysis in Python for Monocytes 
- CertPro

In [2]:
import glob
import os

import anndata
import h5py
import matplotlib
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as scs
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import median_abs_deviation

matplotlib.rcParams["pdf.fonttype"] = 42

In [3]:
# define some color patterns for plotting
nejm_color = [
    "#BC3C29FF",
    "#0072B5FF",
    "#E18727FF",
    "#20854EFF",
    "#7876B1FF",
    "#6F99ADFF",
    "#FFDC91FF",
    "#EE4C97FF",
]
jama_color = [
    "#374E55FF",
    "#DF8F44FF",
    "#00A1D5FF",
    "#B24745FF",
    "#79AF97FF",
    "#6A6599FF",
    "#80796BFF",
]

In [4]:
# define working path
data_path = "/home/workspace/data/ra_longitudinal/scrna/certPro/"
fig_path = "/home/workspace/data/ALTRA_manusript/figures/"
meta_path = "/home/workspace/github/ra-longitudinal/metadata/"
output_path = "/home/workspace/data/ra_longitudinal/output_results/cd16mono/"
# define a project name
proj_name = "RA_lg_converters_scRNA_mono_"
# sc.set_figure_params(fig_path)
sc.settings.figdir = fig_path
sc.settings.autosave = False
sc.set_figure_params(vector_friendly=True, dpi_save=300)

In [5]:
# set fig size
plt.rcParams["figure.figsize"] = [10, 8]

In [6]:
# define helper function
def save_plot(fig, ax, save):
    """
    Save a matplotlib figure to a file.

    Parameters:
    - fig: The matplotlib figure object to be saved.
    - ax: The matplotlib axes object associated with the figure.
    - save: The file path to save the figure to.

    Raises:
    - ValueError: If any of the input parameters are None.

    Returns:
    - None
    """
    if save is not None:
        if ax is not None:
            if fig is not None:
                fig.savefig(save, bbox_inches="tight")
            else:
                raise ValueError("fig is None, cannot save figure.")
        else:
            raise ValueError("ax is None, cannot save figure.")


def filter_limits(df, sign_limit=None, lFCs_limit=None):
    """
    Filter a DataFrame based on sign and log fold change limits.

    Parameters:
    df (DataFrame): The input DataFrame to be filtered.
    sign_limit (float, optional): The limit for p-values. Default is None.
    lFCs_limit (float, optional): The limit for log fold changes. Default is None.

    Returns:
    DataFrame: The filtered DataFrame based on the specified limits.
    """

    # Define limits if not defined
    if sign_limit is None:
        sign_limit = np.inf
    if lFCs_limit is None:
        lFCs_limit = np.inf

    # Filter by absolute value limits
    msk_sign = df["pvals"] < np.abs(sign_limit)
    msk_lFCs = np.abs(df["logFCs"]) < np.abs(lFCs_limit)
    df = df.loc[msk_sign & msk_lFCs]

    return df


def plot_volcano_df(
    data,
    x,
    y,
    top=5,
    genes=None,
    sign_thr=0.05,
    lFCs_thr=0.5,
    sign_limit=None,
    lFCs_limit=None,
    color_pos="#D62728",
    color_neg="#1F77B4",
    color_null="gray",
    figsize=(7, 5),
    dpi=100,
    ax=None,
    return_fig=False,
    save=None,
):
    """
    Plot logFC and p-values from a long formated data-frame.

    Parameters
    ----------
    data : pd.DataFrame
        Results of DEA in long format.
    x : str
        Column name of data storing the logFCs.
    y : str
        Columns name of data storing the p-values.
    top : int
        Number of top differentially expressed features to show.
    sign_thr : float
        Significance threshold for p-values.
    lFCs_thr : float
        Significance threshold for logFCs.
    sign_limit : float
        Limit of p-values to plot in -log10.
    lFCs_limit : float
        Limit of logFCs to plot in absolute value.
    color_pos: str
        Color to plot significant positive genes.
    color_neg: str
        Color to plot significant negative genes.
    color_null: str
        Color to plot rest of the genes.
    figsize : tuple
        Figure size.
    dpi : int
        DPI resolution of figure.
    ax : Axes, None
        A matplotlib axes object. If None returns new figure.
    return_fig : bool
        Whether to return a Figure object or not.
    save : str, None
        Path to where to save the plot. Infer the filetype if ending on {``.pdf``, ``.png``, ``.svg``}.

    Returns
    -------
    fig : Figure, None
        If return_fig, returns Figure object.
    """
    import numpy as np
    from adjustText import adjust_text

    # # Load plotting packages
    # plt = check_if_matplotlib()
    # at = check_if_adjustText()
    # Transform sign_thr
    sign_thr = -np.log10(sign_thr)

    # Extract df
    df = data.copy()
    df["logFCs"] = df[x]
    df["pvals"] = -np.log10(df[y])

    # Filter by limits
    df = filter_limits(df, sign_limit=sign_limit, lFCs_limit=lFCs_limit)

    # Define color by up or down regulation and significance
    df["weight"] = color_null
    up_msk = (df["logFCs"] >= lFCs_thr) & (df["pvals"] >= sign_thr)
    dw_msk = (df["logFCs"] <= -lFCs_thr) & (df["pvals"] >= sign_thr)
    df.loc[up_msk, "weight"] = color_pos
    df.loc[dw_msk, "weight"] = color_neg

    # Plot
    fig = None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi)
    df.plot.scatter(x="logFCs", y="pvals", c="weight", sharex=False, ax=ax)
    ax.set_axisbelow(True)

    # Draw sign lines
    ax.axhline(y=sign_thr, linestyle="--", color="black")
    ax.axvline(x=lFCs_thr, linestyle="--", color="black")
    ax.axvline(x=-lFCs_thr, linestyle="--", color="black")

    if genes is None:
        # Plot top sign features
        signs = df[up_msk | dw_msk].sort_values("pvals", ascending=False)
        signs = signs.iloc[:top]
    else:
        signs = df[up_msk | dw_msk].sort_values("pvals", ascending=False)
        signs = signs.loc[signs.index.isin(genes)]
    # Add labels
    ax.set_ylabel("-log10(p.adj)")
    ax.set_xlabel("Log2 fold change")
    texts = []
    for x, y, s in zip(signs["logFCs"], signs["pvals"], signs.index):
        texts.append(ax.text(x, y, s))
    if len(texts) > 0:
        adjust_text(texts, arrowprops=dict(arrowstyle="-", color="black"), ax=ax)

    save_plot(fig, ax, save)

    if return_fig:
        return fig


# define helper function
def get_marker_expression_stats(adata, markers, cluster_col):
    """
    Calculate mean expression and percent expressing cells for given markers per cluster.

    Parameters
    ----------
    adata : AnnData
        AnnData object with expression data.
    markers : list of str
        List of marker gene names.
    cluster_col : str
        Column in adata.obs to group by (e.g., cluster label).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: cluster_col, gene, mean_expression, percent_expressing
    """
    # Check if cluster_col exists in adata.obs
    if cluster_col not in adata.obs.columns:
        raise ValueError(
            f"'{cluster_col}' not found in adata.obs columns: {list(adata.obs.columns)}"
        )

    # Extract marker expression and cluster info into a DataFrame
    df = sc.get.obs_df(adata, keys=markers + [cluster_col])

    # Define percent expressing function
    def percent_expressing(x):
        return np.mean(x > 0) * 100

    # Aggregate: get mean and percent expressing for each marker per cluster
    agg_df = (
        df.groupby(cluster_col)[markers].agg([percent_expressing, "mean"]).reset_index()
    )

    # If cluster_col is not a column, reset index to make it a column
    if cluster_col not in agg_df.columns:
        agg_df = agg_df.reset_index()

    # Reshape to tidy format:
    # - set cluster_col as index
    # - melt to long format
    # - rename columns for clarity
    # - pivot so each row is cluster/gene, columns are metrics
    result = (
        agg_df.set_index(cluster_col)
        .melt(value_name="value", ignore_index=False)
        .rename(columns={"variable_0": "gene", "variable_1": "metric"})
        .reset_index()
        .pivot(index=[cluster_col, "gene"], columns="metric", values="value")
        .reset_index()
    )

    # Rename columns for clarity
    result = result.rename(
        columns={"mean": "mean_expression", "percent_expressing": "percent_expressing"}
    )

    # Remove index name if present
    result.index.name = None

    return result

# load data

In [4]:
hp.cache_files(["609f7543-d4d5-41e9-a3d2-8e50c3e7c61d"])

['/home/workspace/input/569004694/UCSDCU_Y4/609f7543-d4d5-41e9-a3d2-8e50c3e7c61d/ALTRA_certPro_scRNA_141_samples_combined_adata.h5ad']

In [ ]:
# load the deep clean data
joint_adata_fl = sc.read_h5ad(
   '/home/workspace/input/569004694/UCSDCU_Y4/609f7543-d4d5-41e9-a3d2-8e50c3e7c61d/ALTRA_certPro_scRNA_141_samples_combined_adata.h5ad'
)

In [ ]:
joint_adata_fl.obs["sample.sampleKitGuid"].unique

In [ ]:
# subset the monocytes
joint_adata_fl.obs.loc[
    joint_adata_fl.obs["AIFI_L3_new"].str.contains("monocyte"), "AIFI_L3_new"
].unique()

In [ ]:
# subset aim3 data
aim3_meta = pd.read_csv(
    meta_path + "ALTRA_RA_Aim3_ALTRA_converters_longitudinal_scrna_metadata.csv"
)
aim3_meta.columns

In [ ]:
# subset the monocytes in aim3
mono_adata = joint_adata_fl[
    (joint_adata_fl.obs["AIFI_L3_new"].str.contains("monocyte"))
].copy()

In [ ]:
mono_adata

## Rerun basic normalization

In [ ]:
mono_adata.obs["sample.sampleKitGuid"].unique()

In [ ]:
# save the raw counts
mono_adata.layers["counts"] = mono_adata.X.copy()

In [ ]:
# mitochondrial genes
mono_adata.var["mt"] = mono_adata.var_names.str.startswith("MT-")
# ribosomal genes
mono_adata.var["ribo"] = mono_adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
mono_adata.var["hb"] = mono_adata.var_names.str.contains(("^HB[^(P)]"))
sc.pp.calculate_qc_metrics(
    mono_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)

In [ ]:
# cpm normalization
sc.pp.normalize_total(mono_adata, target_sum=1e4, inplace=True)
sc.pp.log1p(mono_adata)

In [ ]:
# %%time
sc.pp.highly_variable_genes(mono_adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
mono_adata.raw = mono_adata

In [ ]:
mono_adata.raw.X

In [ ]:
sc.pp.scale(mono_adata, max_value=10)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(mono_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
sc.pl.pca_scatter(mono_adata, color=["pct_counts_ribo", "pct_counts_mt"])

In [ ]:
# # setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
# sc.pp.pca(mono_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(mono_adata, log=True)

In [ ]:
sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=20, use_rep="X_pca")
sc.tl.umap(mono_adata)

In [ ]:
# save the pre-harmonaized umap
mono_adata.obsm["X_orignial_umap"] = mono_adata.obsm["X_umap"].copy()

In [ ]:
mono_adata

In [ ]:
# run harmony
import scanpy.external as sce

sce.pp.harmony_integrate(
    mono_adata,
    ["file.batchID", "subject.biologicalSex"],
    adjusted_basis="X_pca_harmony",
)
sc.pp.neighbors(mono_adata, n_neighbors=30, n_pcs=20, use_rep="X_pca_harmony")
sc.tl.umap(mono_adata)
mono_adata.obsm["X_harmony_umap"] = mono_adata.obsm["X_umap"].copy()

### run TSNE

In [ ]:
# sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.tsne(mono_adata, use_rep="X_pca_harmony")

In [ ]:
mono_adata.obsm["X_pca_harmony"]

In [ ]:
# mono_adata.obsm['X_harmony_umap'] = mono_adata.obsm['X_umap'].copy()

In [ ]:
sc.pl.pca_loadings(mono_adata, components="1,2,3")

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_0_5", resolution=0.5)

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_1", resolution=1, n_iterations=2)

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_1_2", resolution=1.2, n_iterations=2)

In [ ]:
mono_adata

In [ ]:
sc.pl.umap(
    mono_adata,
    color=[
        "file.batchID",
        "subject.subjectGuid",
        "leiden_0_5",
        "subject.biologicalSex",
        "AIFI_L2",
        "AIFI_L3_new",
    ],
    ncols=3,
    legend_loc="on data",
    wspace=0.4,
    save=proj_name + "_rna_hamony_umap.png",
)

In [ ]:
# save data
mono_adata.write_h5ad(data_path + "ALTRA_scRNA_all_monocytes_certPro.h5ad")

### upload data to HISE

In [2]:
import hisepy as hp

In [6]:
# upload file to hise
hp.upload.upload_files(
    files=[
        "/home/workspace/data/ALTRA_manusript/ALTRA_scRNA_all_monocytes_certPro.h5ad"
    ],
    study_space_id="223de760-9624-45bd-aefe-ca24c75b1800",
    title="AlTRA scRNA certpro monocytes h5ad object",
    input_file_ids=["609f7543-d4d5-41e9-a3d2-8e50c3e7c61d"],
    destination="certpro_scrna",
)

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '469e04ba-5361-42dd-83ec-802ae43ab5a2',
 'ProcessId': '957dbcc8-447f-4223-830f-99e1c2d19c2d',
 'WorkflowId': 'c1d83b5e-afef-46d7-89e2-40d1895c7af0',
 'FileIds': ['6189538f-bae1-4f6e-b219-d322a530e65f']}

# Session Info

In [ ]:
import sinfo

sinfo.sinfo(write_req_file=False)